## ETAPA 4 - Streaming de ventas con Spark

**Qué hacer:** Con `spark.readStream` lean una carpeta donde van apareciendo lotes de ventas, calculen Total, agrupen por país (y apliquen un filtro, por ejemplo `Quantity > 100`). Escriban el resultado con `writeStream` a memoria y muéstrenlo. 

In [4]:
import os
import time
import shutil

hadoop_dir = os.path.abspath("hadoop")
if os.path.exists(hadoop_dir):
    os.environ["HADOOP_HOME"] = hadoop_dir
    os.environ["PATH"] = os.path.join(hadoop_dir, "bin") + ";" + os.environ.get("PATH", "")

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, round, sum as _sum

spark = SparkSession.builder \
    .appName("Etapa4_StreamingVentas") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

base_dir = os.path.abspath(".")
input_dir = os.path.join(base_dir, "datos_streaming")
checkpoint_dir = os.path.join(base_dir, "checkpoints/ventas_streaming")

for d in [input_dir, checkpoint_dir]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d)

schema = StructType([
    StructField("Invoice", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", StringType(), True),
    StructField("Price", DoubleType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Country", StringType(), True)
])

streaming_df = spark.readStream \
    .option("header", "true") \
    .schema(schema) \
    .csv(input_dir)

ventas_filtradas = streaming_df \
    .filter(col("Quantity") > 100) \
    .withColumn("Total", round(col("Quantity") * col("Price"), 2))

ventas_por_pais = ventas_filtradas \
    .groupBy("Country") \
    .agg(
        _sum("Quantity").alias("Unidades_Vendidas"),
        round(_sum("Total"), 2).alias("Total_Ventas")
    )

query = ventas_por_pais.writeStream \
    .queryName("ventas_en_memoria") \
    .outputMode("complete") \
    .format("memory") \
    .option("checkpointLocation", checkpoint_dir) \
    .start()

print("Streaming iniciado correctamente. Escuchando carpeta:", input_dir)

Streaming iniciado correctamente. Escuchando carpeta: c:\Users\JGuerra\Taller-_Streaming_Kafka\datos_streaming


In [5]:
lote1 = """Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,CHRISTMAS BALL,120,2009-12-01 07:45:00,6.95,13085,United Kingdom
489435,22350,CAT BOWL,150,2009-12-01 07:46:00,2.55,13085,United Kingdom
489436,48173C,DOOR MAT,200,2009-12-01 09:06:00,5.95,13078,France
489437,22111,CONFECTIONERY,80,2009-12-01 09:10:00,1.25,15362,Germany
"""

with open(os.path.join(input_dir, "lote_01.csv"), "w", encoding="utf-8") as f:
    f.write(lote1.strip())

time.sleep(3)
query.processAllAvailable()

print(">>> RESULTADOS TRAS EL LOTE 1 (PANTALLAZO 1):")
spark.sql("SELECT * FROM ventas_en_memoria ORDER BY Total_Ventas DESC").show()

>>> RESULTADOS TRAS EL LOTE 1 (PANTALLAZO 1):
+--------------+-----------------+------------+
|       Country|Unidades_Vendidas|Total_Ventas|
+--------------+-----------------+------------+
|United Kingdom|              270|      1216.5|
|        France|              200|      1190.0|
+--------------+-----------------+------------+



In [6]:
lote2 = """Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489500,85048,CHRISTMAS BALL,250,2009-12-01 10:00:00,6.95,14000,France
489501,21232,CERAMIC BOX,180,2009-12-01 10:15:00,1.25,15000,Germany
489502,22041,RECORD FRAME,300,2009-12-01 10:30:00,2.10,13085,United Kingdom
"""

with open(os.path.join(input_dir, "lote_02.csv"), "w", encoding="utf-8") as f:
    f.write(lote2.strip())

time.sleep(3)
query.processAllAvailable()

print(">>> RESULTADOS TRAS EL LOTE 2 (PANTALLAZO 2):")
spark.sql("SELECT * FROM ventas_en_memoria ORDER BY Total_Ventas DESC").show()

query.stop()

>>> RESULTADOS TRAS EL LOTE 2 (PANTALLAZO 2):
+--------------+-----------------+------------+
|       Country|Unidades_Vendidas|Total_Ventas|
+--------------+-----------------+------------+
|        France|              450|      2927.5|
|United Kingdom|              570|      1846.5|
|       Germany|              180|       225.0|
+--------------+-----------------+------------+

